In [ ]:
# ======================================================================
#  CTP-FMQR: censored two-part finite mixture quantile regression for
#  zero-inflated and right-censored longitudinal data.
#
#  Reproduces all tables, figures and LaTeX macros of the paper:
#    data construction -> numerical verification -> simulation ->
#    selection of G -> bootstrap coverage -> applications (ICI, RAND HIE)
#    -> sensitivity to the number of components -> LaTeX output.
#
#  Usage
#    CTPFMQR_MODE=SMOKE    python ctpfmqr_pipeline.py   (about 20-30 min, checks the pipeline)
#    CTPFMQR_MODE=STANDARD python ctpfmqr_pipeline.py   (results reported in the paper)
#  In Google Colab, set the variable before running, e.g.
#    import os; os.environ["CTPFMQR_MODE"] = "STANDARD"
#  Long parts can be split across sessions with PARTS and SIM_ERRS_OVERRIDE;
#  every part resumes from its checkpoints. Finish with PARTS['report'] = True.
#
#  Data: the ISO 1000 raw file is not distributed (see data/README.md);
#  the RAND HIE data are downloaded automatically.
# ======================================================================
import os

# ----------------------------------------------------------------------
# 0. SETTINGS
# ----------------------------------------------------------------------
VERSION = "v5.1"
MODE = os.environ.get("CTPFMQR_MODE", "SMOKE").upper()   # "SMOKE" | "QUICK" | "STANDARD" | "FULL"
PARTS = dict(
    verify   = True,        # analytic/numerical self-tests (about 1 min)
    data     = True,        # build the ICI panel from the raw xlsx file
    sim      = True,        # main simulation grid (R1)
    gsel     = True,        # selection of G (R2)
    coverage = True,        # bootstrap coverage
    ici      = True,        # Application I: ICI exports (+ robustness)
    rhie     = True,        # Application II: RAND HIE
    report   = True,        # tables, figures, G sensitivity, numbers.tex, manifest, zip
)
SIM_ERRS_OVERRIDE = None    # e.g. ['E3'] to split the simulation across sessions
USE_DRIVE = True            # Google Drive in Colab; local folder otherwise
OUT_ROOT = "ctpfmqr_v5"
N_JOBS = -1
BOOT_METHOD = "cluster"     # "cluster" (recommended) or "parametric"
COV_METHODS = ("cluster",)  # add "parametric" to document its behaviour

# --- ICI data construction ---------------------------------------------
ICI_RAW_FILE = "iso_1000_1997_2012.xlsx"
ICI_MIN_YEARS = 10          # firms observed in at least this many years

# --- Application I: ICI exports ------------------------------------------
ICI_SPEC = dict(
    name="ici", file="ici_panel.csv", url=None,
    id_col="fid", time_col="year", y_col="EXP",
    covs=["SIZE", "PROD", "KINT", "lATR", "FRI", "TREND"],
    log_covs=[], add_trend=False, years=None, balanced=False, filter_col=None,
    thr_base="positive", main_kappa=0.20,
    labels=dict(SIZE="SIZE", PROD="PROD", KINT="KINT", lATR="log ATR", FRI="FRI", TREND="TREND"),
    y_label="EXP", xlabel="exports (thousand USD)", section="Application I (ICI)",
    main_rows=None, intercept=True, supp=False,
)
ICIM_SPEC = dict(ICI_SPEC, name="icimanuf", filter_col="MANUF",
                 section="Application I robustness: manufacturing firms (supplement)", supp=True)
ICIB_SPEC = dict(ICI_SPEC, name="icibal", years=(1997, 2012), balanced=True,
                 section="Application I robustness: balanced panel 1997-2012 (supplement)", supp=True)

# --- Application II: RAND HIE ----------------------------------------------
RHIE_SPEC = dict(
    name="rhie", file="RandHIE.csv",
    url="https://vincentarelbundock.github.io/Rdatasets/csv/sampleSelection/RandHIE.csv",
    id_col="zper", time_col=None, y_col="meddol",
    covs=['logc', 'idp', 'lpi', 'fmde', 'physlm', 'disea', 'hlthg', 'hlthf', 'hlthp',
          'linc', 'lfam', 'educdec', 'xage', 'female', 'child', 'fchild', 'black'],
    log_covs=[], add_trend=False, years=None, balanced=False, filter_col=None,
    thr_base="positive", main_kappa=0.10,
    labels=dict(logc='LC', idp='IDP', lpi='LPI', fmde='FMDE', physlm='PHYSLIM', disea='NDISEASE',
                hlthg='HLTHG', hlthf='HLTHF', hlthp='HLTHP', linc='LINC', lfam='LFAM',
                educdec='EDUCDEC', xage='AGE', female='FEMALE', child='CHILD',
                fchild='FEMCHILD', black='BLACK'),
    y_label="MEDDOL", xlabel="medical expenditure (USD)", section="Application II (RHIE)",
    main_rows=['logc', 'idp', 'disea', 'physlm', 'linc'], intercept=False, supp=False,
)
APP_SPECS = [ICI_SPEC, ICIM_SPEC, ICIB_SPEC, RHIE_SPEC]

for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ.setdefault(_v, "1")
import re, time, zlib, json, pickle, glob, shutil, platform, traceback, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import optimize, stats, sparse, integrate
from scipy.special import expit, log_expit, logsumexp
from sklearn.cluster import KMeans
from joblib import Parallel, delayed
warnings.filterwarnings("ignore")

ERRS_ALL = ['E1', 'E2', 'E3', 'E4']
NT_ALL = [(100, 5), (100, 10), (250, 5), (250, 10)]
MODES = {
    "SMOKE":    dict(sim_B=2, sim_errs=['E1', 'E4'], sim_nt=[(100, 5)],
                     gsel_B=2, gsel_Ns=(100,), cov_R=2, cov_B=10,
                     app_Gmax=2, app_Gcap=3, app_starts=1, app_B=5,
                     ici_kappas=[0.20], rhie_kappas=[0.10], rhie_sub=300, ici_robust=False),
    "QUICK":    dict(sim_B=20, sim_errs=ERRS_ALL, sim_nt=NT_ALL,
                     gsel_B=20, gsel_Ns=(100, 250), cov_R=20, cov_B=50,
                     app_Gmax=5, app_Gcap=7, app_starts=3, app_B=50,
                     ici_kappas=[0.10, 0.20, 0.30], rhie_kappas=[0.10], rhie_sub=None, ici_robust=False),
    "STANDARD": dict(sim_B=50, sim_errs=ERRS_ALL, sim_nt=NT_ALL,
                     gsel_B=50, gsel_Ns=(100, 250), cov_R=50, cov_B=100,
                     app_Gmax=7, app_Gcap=10, app_starts=5, app_B=100,
                     ici_kappas=[0.10, 0.20, 0.30], rhie_kappas=[0.05, 0.10, 0.20], rhie_sub=None,
                     ici_robust=True),
    "FULL":     dict(sim_B=100, sim_errs=ERRS_ALL, sim_nt=NT_ALL,
                     gsel_B=100, gsel_Ns=(100, 250), cov_R=100, cov_B=200,
                     app_Gmax=7, app_Gcap=12, app_starts=10, app_B=200,
                     ici_kappas=[0.10, 0.20, 0.30], rhie_kappas=[0.05, 0.10, 0.20], rhie_sub=None,
                     ici_robust=True),
}
if MODE not in MODES:
    raise ValueError(f'CTPFMQR_MODE must be one of {list(MODES)}, got {MODE!r}')
M = MODES[MODE]

# --- folders -----------------------------------------------------------
ROOT = None
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        ROOT = f"/content/drive/MyDrive/{OUT_ROOT}"
    except Exception as e:
        print("Drive not available, using a local folder:", e)
if ROOT is None:
    ROOT = f"./{OUT_ROOT}"
SIM_DIR, APP_DIR, DATA_DIR = (os.path.join(ROOT, d) for d in ("sim", "app", "data"))
RES_DIR = os.path.join(ROOT, "results")
CSV_DIR, PNG_DIR = os.path.join(RES_DIR, "csv"), os.path.join(RES_DIR, "png")
OVL_DIR = os.path.join(ROOT, "overleaf")
TAB_DIR = os.path.join(OVL_DIR, "tables")
SUPP_DIR = os.path.join(TAB_DIR, "supp")
FIG_DIR = os.path.join(OVL_DIR, "figures")
for d in (SIM_DIR, APP_DIR, DATA_DIR, CSV_DIR, PNG_DIR, TAB_DIR, SUPP_DIR, FIG_DIR):
    os.makedirs(d, exist_ok=True)
NCPU = os.cpu_count() or 1
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 300, "font.family": "serif",
                     "axes.spines.top": False, "axes.spines.right": False})

# --- logging and output registry ----------------------------------------
LOG_FN = os.path.join(RES_DIR, "run_log.txt")
INDEX_FN = os.path.join(RES_DIR, "OUTPUT_INDEX.csv")
STATUS = {}

def log(msg):
    s = f"[{time.strftime('%Y-%m-%d %H:%M:%S')}] {msg}"
    print(s, flush=True)
    with open(LOG_FN, "a") as f:
        f.write(s + "\n")

def banner(t):
    print("\n" + "=" * 90 + f"\n{t}\n" + "=" * 90, flush=True)
    log(t)

def _load_index():
    if os.path.exists(INDEX_FN):
        return pd.read_csv(INDEX_FN)
    return pd.DataFrame(columns=['section', 'kind', 'path', 'latex', 'description', 'mode', 'updated'])

def register(kind, path, latex, desc, section):
    idx = _load_index()
    idx = idx[idx.path != path]
    row = dict(section=section, kind=kind, path=path, latex=latex, description=desc,
               mode=MODE, updated=time.strftime('%Y-%m-%d %H:%M'))
    idx = pd.concat([idx, pd.DataFrame([row])], ignore_index=True)
    idx.sort_values(['section', 'path']).to_csv(INDEX_FN, index=False)

def save_table(name, colspec, header, rows, desc, section, df=None, supp=False):
    d = SUPP_DIR if supp else TAB_DIR
    rel = f"tables/{'supp/' if supp else ''}{name}"
    body = ("\\begin{tabular}{" + colspec + "}\n\\toprule\n" + header + " \\\\\n\\midrule\n"
            + "\n".join(rows) + "\n\\bottomrule\n\\end{tabular}\n")
    with open(os.path.join(d, name + ".tex"), "w") as f:
        f.write(body)
    if df is not None:
        df.to_csv(os.path.join(CSV_DIR, "tab_" + name.replace("tab_", "") + ".csv"), index=False)
    register("table", rel + ".tex", f"\\input{{{rel}}}", desc, section)
    print(f"  [table]  {rel}.tex")

def save_figure(fig, name, desc, section, width=r"\textwidth"):
    fig.savefig(os.path.join(FIG_DIR, name + ".pdf"), bbox_inches="tight")
    fig.savefig(os.path.join(PNG_DIR, name + ".png"), dpi=200, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    register("figure", f"figures/{name}.pdf",
             f"\\includegraphics[width={width}]{{figures/{name}}}", desc, section)
    print(f"  [figure] figures/{name}.pdf")

def save_config():
    cfg = dict(VERSION=VERSION, MODE=MODE, M={k: (list(v) if isinstance(v, (list, tuple)) else v)
                                              for k, v in M.items()},
               PARTS=PARTS, SIM_ERRS_OVERRIDE=SIM_ERRS_OVERRIDE, BOOT_METHOD=BOOT_METHOD,
               COV_METHODS=list(COV_METHODS), ICI_RAW_FILE=ICI_RAW_FILE, ICI_MIN_YEARS=ICI_MIN_YEARS,
               APP_SPECS=APP_SPECS, NCPU=NCPU,
               versions=dict(python=platform.python_version(), numpy=np.__version__,
                             pandas=pd.__version__, scipy=__import__('scipy').__version__,
                             sklearn=__import__('sklearn').__version__))
    with open(os.path.join(RES_DIR, f"RUN_CONFIG_{MODE}.json"), "w") as f:
        json.dump(cfg, f, indent=2, default=str)

def _fi(x):
    return f'{x:,.0f}'.replace(',', '{,}')

def _ff(x, d=2):
    return r'\tbd' if x is None or not np.isfinite(x) else f'{x:.{d}f}'

def _f3(x):
    return '--' if x is None or not np.isfinite(x) else f'{x:.3f}'

def _tex(s):
    return str(s).replace('<', '$<$').replace('>', '$>$').replace('_', r'\_')

print(f"{VERSION} | MODE={MODE} | CPU={NCPU} | output: {ROOT}")
print("Parts in this session:", [k for k, v in PARTS.items() if v])

# ----------------------------------------------------------------------
# 1. ICI DATA CONSTRUCTION (raw ISO 1000 file -> analysis panel)
# ----------------------------------------------------------------------
ICI_PANEL_FN = os.path.join(DATA_DIR, ICI_SPEC['file'])
ICI_ALL_FN = os.path.join(DATA_DIR, 'ici_panel_all_years.csv')
ICI_META_FN = os.path.join(DATA_DIR, 'ici_data_meta.json')
ICI_REPORT_FN = os.path.join(RES_DIR, 'ici_data_report.txt')
ICI_VIF_FN = os.path.join(CSV_DIR, 'ici_vif_raw.csv')
ICI_PANELS_FN = os.path.join(CSV_DIR, 'ici_panels_raw.csv')
SET_A = ['GVA', 'CAP', 'EMP', 'ATR', 'FRI', 'TREND']       # level-based specification (robustness)
SET_B = ['SIZE', 'PROD', 'KINT', 'lATR', 'FRI', 'TREND']   # ratio-based specification (main)
_TR = str.maketrans('ÇĞİÖŞÜçğıöşü', 'CGIOSUcgiosu')
_STOP = (r'\b(A\s?S|AS|ANONIM|ORTAKLIGI|SIRKETI|SAN|SANAYI|SANAYII|VE|TIC|TICARET|'
         r'LTD|STI|T\s?A\s?S|TAS|INC|CO|GEN|MUD|MUESS|ISL|ISLETMELERI)\b')

def _norm_name(s):
    s = str(s).translate(_TR).upper()
    s = re.sub(r'[^A-Z0-9 ]', ' ', s)
    s = re.sub(_STOP, ' ', s)
    return re.sub(r'\s+', ' ', s).strip()

def _vif(X):
    C = np.corrcoef(X.values, rowvar=False)
    return pd.Series(np.diag(np.linalg.inv(C)), index=X.columns)

def build_ici_panel():
    fn = os.path.join(DATA_DIR, ICI_RAW_FILE)
    if not os.path.exists(fn):
        raise FileNotFoundError(f'{fn} not found. Put the raw ISO 1000 file in {DATA_DIR}.')
    rep = []

    def say(*a):
        s = ' '.join(str(x) for x in a)
        print(s)
        rep.append(s)

    raw = pd.read_excel(fn, engine='openpyxl')
    raw.columns = [str(c).strip() for c in raw.columns]
    num = ['year', 'firmcode', 'nacerev2', 'fiyat', 'output', 'reveneu', 'valueadded',
           'capital', 'netactives', 'profits', 'exp', 'employment', 'public', 'private', 'foreign']
    miss = [c for c in num + ['firmname'] if c not in raw.columns]
    if miss:
        raise KeyError(f'raw ICI file: columns missing {miss}. Available: {list(raw.columns)}')
    for c in num:
        raw[c] = pd.to_numeric(raw[c], errors='coerce')
    raw['firmname'] = raw['firmname'].astype(str).str.strip().str.upper()
    meta = dict(raw_rows=int(len(raw)), min_years=ICI_MIN_YEARS)
    d = raw[~raw['firmname'].isin(['-', 'NAN', '']) & raw['year'].notna()].copy().reset_index(drop=True)
    meta['named_rows'] = int(len(d))
    d['nkey'] = d['firmname'].map(_norm_name)
    d['tok'] = d['nkey'].str.split().str[0]
    n_empty = int((d['nkey'] == '').sum())
    if n_empty:
        say(f'[WARNING] {n_empty} records have an empty normalised name')

    # hybrid identifier: code + first word OR normalised name; no same-year merges
    parent, years = {}, {}

    def add(x, yrs=()):
        if x not in parent:
            parent[x] = x
            years[x] = set(yrs)

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(a, b):
        ra, rb = find(a), find(b)
        if ra == rb:
            return True
        if years[ra] & years[rb]:
            return False
        parent[rb] = ra
        years[ra] |= years[rb]
        return True

    blocked = 0
    order = d.sort_values(['year', 'output'], ascending=[True, False]).index
    for i in order:
        r = d.loc[i]
        rid, nn = f'R{i}', f'N|{r.nkey}'
        cn = f'C|{int(r.firmcode)}|{r.tok}' if pd.notna(r.firmcode) else None
        add(rid, [int(r.year)])
        add(nn)
        ok1, ok2 = union(rid, nn), True
        if cn:
            add(cn)
            ok2 = union(rid, cn)
        blocked += (not ok1) + (not ok2)
    d['fid'] = [find(f'R{i}') for i in d.index]
    fid_map = {f: f'F{k:05d}' for k, f in enumerate(d['fid'].unique())}
    d['fid'] = d['fid'].map(fid_map)
    assert not d.duplicated(['fid', 'year']).any()
    meta.update(firms_code=int(d.firmcode.nunique()), firms_name=int(d.nkey.nunique()),
                firms_hybrid=int(d.fid.nunique()), blocked_links=int(blocked))
    say(f'Firms: code = {meta["firms_code"]}, name = {meta["firms_name"]}, hybrid = {meta["firms_hybrid"]}')
    say(f'Links not made because of same-year conflicts: {blocked}')

    # cleaning and variables
    for c in ['output', 'reveneu', 'valueadded', 'capital', 'netactives']:
        d.loc[d[c] < 1000, c] = np.nan            # values in (0, 1) are corrupted negatives
    d.loc[d['employment'] <= 0, 'employment'] = np.nan
    fin_ok = d[['output', 'netactives', 'employment']].notna().all(axis=1)
    d = d[~(d['exp'].isna() & ~fin_ok)].copy()    # blank exports and missing financials: missing
    meta['clean_rows'] = int(len(d))
    d['EXP_src'] = np.where(d['exp'].isna(), 'blank', np.where(d['exp'] == 0, 'zero', 'positive'))
    d['EXP'] = d['exp'].fillna(0.0)
    d = d.sort_values(['fid', 'year'])
    prev = d.groupby('fid')[['year', 'netactives']].shift(1)
    consec = (prev['year'] == d['year'] - 1) & prev['netactives'].notna()
    d['avg_assets'] = np.where(consec, (d['netactives'] + prev['netactives']) / 2, d['netactives'])
    d['OUT_r'] = d['output'] / d['fiyat'] * 100
    d['CAP_r'] = d['capital'] / d['fiyat'] * 100
    d['GVA_r'] = d['valueadded'] / d['fiyat'] * 100
    with np.errstate(divide='ignore', invalid='ignore'):
        d['SIZE'] = np.log(d['employment'])
        d['PROD'] = np.log(d['OUT_r'] / d['employment'])
        d['KINT'] = np.log(d['CAP_r'] / d['employment'])
        d['lATR'] = np.log(d['reveneu'] / d['avg_assets'])
        d['GVA'] = np.log(d['GVA_r'])
        d['CAP'] = np.log(d['CAP_r'])
    d['FRI'] = d['foreign'] / 100
    d['TREND'] = d['year'] - 1997
    d['EMP'] = d['SIZE']
    d['ATR'] = d['reveneu'] / d['avg_assets']
    d['MANUF'] = d['nacerev2'].between(10, 33)

    # multicollinearity of the two covariate sets
    vifs = {}
    for lab, S in (('A', SET_A), ('B', SET_B)):
        ok = d[S].replace([np.inf, -np.inf], np.nan).notna().all(axis=1)
        vifs[lab] = _vif(d.loc[ok, S])
        say(f'Set {lab}: complete records = {int(ok.sum())}')
        say(vifs[lab].round(2).to_string())
    pd.DataFrame({'setA': SET_A, 'vifA': vifs['A'].values,
                  'setB': SET_B, 'vifB': vifs['B'].values}).to_csv(ICI_VIF_FN, index=False)
    meta.update(vif_max_A=float(vifs['A'].max()), vif_max_B=float(vifs['B'].max()))

    # candidate panels (Set B)
    okB = d[SET_B].replace([np.inf, -np.inf], np.nan).notna().all(axis=1)
    P = d[okB].copy()
    meta['complete_rows'] = int(len(P))
    res = []
    for scope, Q0 in (('All sectors', P), ('Manufacturing (NACE 10-33)', P[P.MANUF])):
        ny = Q0.groupby('fid')['year'].transform('nunique')
        for m in (5, 8, 10, 12, 16):
            Q = Q0[ny >= m]
            nf = Q.fid.nunique()
            res.append(dict(scope=scope, min_years=m, firms=nf, obs=len(Q),
                            mean_T=len(Q) / max(nf, 1), zero_share=float((Q.EXP == 0).mean())))
    Pn = pd.DataFrame(res)
    Pn.to_csv(ICI_PANELS_FN, index=False)
    say(Pn.round(3).to_string(index=False))
    meta['bal_firms'] = int(Pn[(Pn.scope == 'All sectors') & (Pn.min_years == 16)].firms.iloc[0])

    # final panel
    ny = P.groupby('fid')['year'].transform('nunique')
    F = P[ny >= ICI_MIN_YEARS].copy()
    zeros = F.EXP == 0
    meta.update(final_firms=int(F.fid.nunique()), final_obs=int(len(F)),
                zero_share=float(zeros.mean()),
                zero_blank_share=float((F.loc[zeros, 'EXP_src'] == 'blank').mean()) if zeros.any() else 0.0,
                nonmanuf_share=float((~F.MANUF).mean()),
                year_min=int(F.year.min()), year_max=int(F.year.max()))
    say(f'Final panel: {meta["final_firms"]} firms, {meta["final_obs"]} obs, '
        f'zero share {meta["zero_share"]:.3f} (blank in source: {meta["zero_blank_share"]:.3f}), '
        f'non-manufacturing share {meta["nonmanuf_share"]:.3f}')
    keep = ['fid', 'firmname', 'firmcode', 'year', 'nacerev2', 'MANUF', 'EXP', 'EXP_src'] + SET_B + \
           ['GVA', 'CAP', 'ATR', 'OUT_r', 'CAP_r', 'employment', 'fiyat']
    F[keep].to_csv(ICI_PANEL_FN, index=False)
    d[keep].to_csv(ICI_ALL_FN, index=False)
    with open(ICI_META_FN, 'w') as f:
        json.dump(meta, f, indent=2)
    with open(ICI_REPORT_FN, 'w', encoding='utf-8') as f:
        f.write('\n'.join(rep))
    log(f'ICI panel written: {ICI_PANEL_FN}')
    return meta

def report_ici_data():
    if not os.path.exists(ICI_META_FN):
        return
    m = json.load(open(ICI_META_FN))
    sec = 'Application I (ICI): data construction (supplement)'
    rows = [f'Records in the raw file & {_fi(m["raw_rows"])} \\\\',
            f'Records with a firm name & {_fi(m["named_rows"])} \\\\',
            f'Distinct firm codes & {_fi(m["firms_code"])} \\\\',
            f'Distinct normalised firm names & {_fi(m["firms_name"])} \\\\',
            f'Firms (hybrid identifier) & {_fi(m["firms_hybrid"])} \\\\',
            f'Records after cleaning & {_fi(m["clean_rows"])} \\\\',
            f'Records with complete covariates & {_fi(m["complete_rows"])} \\\\',
            f'Firms observed in at least {m["min_years"]} years & {_fi(m["final_firms"])} \\\\',
            f'Firm-year observations in the analysis panel & {_fi(m["final_obs"])} \\\\',
            f'Zero exports (\\%) & {100 * m["zero_share"]:.1f} \\\\',
            f'\\quad of which blank in the source (\\%) & {100 * m["zero_blank_share"]:.1f} \\\\',
            f'Non-manufacturing observations (\\%) & {100 * m["nonmanuf_share"]:.1f} \\\\']
    save_table('ici_construction', 'lr', 'Step & Count', rows,
               'ICI: construction of the analysis panel', sec, supp=True)
    if os.path.exists(ICI_VIF_FN):
        V = pd.read_csv(ICI_VIF_FN)
        labA = {'GVA': 'log GVA', 'CAP': 'log CAP', 'EMP': 'log EMP', 'ATR': 'ATR', 'FRI': 'FRI', 'TREND': 'TREND'}
        labB = ICI_SPEC['labels']
        rows = [f'{labA.get(r.setA, r.setA)} & {r.vifA:.2f} & {labB.get(r.setB, r.setB)} & {r.vifB:.2f} \\\\'
                for r in V.itertuples()]
        save_table('ici_vif', 'lrlr', r'Set A & VIF & Set B & VIF', rows,
                   'ICI: variance inflation factors of the two covariate sets', sec, df=V, supp=True)
    if os.path.exists(ICI_PANELS_FN):
        Pn = pd.read_csv(ICI_PANELS_FN)
        rows = [f'{r.scope} & {r.min_years} & {_fi(r.firms)} & {_fi(r.obs)} & {r.mean_T:.1f} & '
                f'{100 * r.zero_share:.1f} \\\\' for r in Pn.itertuples()]
        save_table('ici_panels', 'lrrrrr',
                   r'Scope & Min.\ years & Firms & Obs. & Mean $T$ & Zeros (\%)', rows,
                   'ICI: candidate panels by minimum number of years', sec, df=Pn, supp=True)

# ----------------------------------------------------------------------
# 2. MODEL CORE
#   M1 CTP-FMQR (censored AL) | M2 ignores censoring | M3 discards censored
#   M4 two-part Gaussian Tobit (NPML) | Oracle: model fitted to uncensored data
# ----------------------------------------------------------------------
class PanelData:
    """Semi-continuous, right-censored longitudinal data."""
    def __init__(self, uid, y, S, X, c=None):
        self.uid_raw = np.asarray(uid)
        self.uid, _ = pd.factorize(self.uid_raw)
        self.y = np.asarray(y, float)
        n = self.y.size
        self.S = np.asarray(S, float).reshape(n, -1)
        self.X = np.asarray(X, float).reshape(n, -1)
        self.c = (np.full(n, np.inf) if c is None
                  else np.broadcast_to(np.asarray(c, float), (n,)).copy())
        self.ystar = np.minimum(self.y, self.c)
        self.d = self.ystar <= 0
        self.delta = self.y < self.c
        self.ly = np.log(np.where(self.ystar > 0, self.ystar, 1.0))
        with np.errstate(divide='ignore'):
            self.lc = np.log(self.c)
        self.n, self.N = n, int(self.uid.max()) + 1
        self.M = sparse.csr_matrix((np.ones(n), (self.uid, np.arange(n))), shape=(self.N, n))

    def subset(self, mask):
        return PanelData(self.uid_raw[mask], self.y[mask], self.S[mask], self.X[mask], self.c[mask])

    def naive(self):
        return PanelData(self.uid_raw, self.ystar, self.S, self.X, None)


def make_variant(D, method):
    if method in ('M1', 'M4'):
        return D
    if method == 'M2':
        return D.naive()
    if method == 'M3':
        return D.subset(D.delta)
    raise ValueError(method)


def _rho(u, tau):
    return u * (tau - (u < 0))

def pos_logdens(fam, tau, r, s):
    u = r / s
    if fam == 'al':
        return np.log(tau * (1 - tau)) - np.log(s) - _rho(u, tau)
    return -0.5 * np.log(2 * np.pi) - np.log(s) - 0.5 * u * u

def pos_logsurv(fam, tau, rc, s):
    u = rc / s
    if fam == 'al':
        um = np.minimum(u, 0.0)
        return np.where(u <= 0, np.log1p(-tau * np.exp((1 - tau) * um)), np.log(1 - tau) - tau * u)
    return stats.norm.logsf(u)

def pos_dmu_dens(fam, tau, r, s):
    u = r / s
    return ((tau - (u < 0)) if fam == 'al' else u) / s

def pos_dmu_surv(fam, tau, rc, s):
    u = rc / s
    if fam == 'al':
        e = np.exp((1 - tau) * np.minimum(u, 0.0))
        h = np.where(u <= 0, tau * (1 - tau) * e / (1 - tau * e), tau)
    else:
        h = np.exp(stats.norm.logpdf(u) - stats.norm.logsf(u))
    return h / s

def comp_loglik(D, th, tau, fam):
    """n x G matrix of log contributions, eq. (5), including the Jacobian 1/y."""
    eta = (D.S @ th['gamma'])[:, None] + th['a0'][None, :]
    ll = np.where(D.d[:, None], log_expit(eta), log_expit(-eta))
    mu = (D.X @ th['beta'])[:, None] + th['a1'][None, :]
    unc = (~D.d) & D.delta
    cen = (~D.d) & (~D.delta)
    s = float(th['sigma'])
    if unc.any():
        ll[unc] += pos_logdens(fam, tau, D.ly[unc, None] - mu[unc], s) - D.ly[unc, None]
    if cen.any():
        ll[cen] += pos_logsurv(fam, tau, D.lc[cen, None] - mu[cen], s)
    return ll

def estep(D, th, tau, fam):
    A = D.M @ comp_loglik(D, th, tau, fam) + np.log(th['pi'])[None, :]
    lse = logsumexp(A, axis=1)
    return np.exp(A - lse[:, None]), float(lse.sum())

def mstep_binary(D, th, W, n_newton=2):
    S, p, G = D.S, D.S.shape[1], W.shape[1]
    g, a0 = th['gamma'].copy(), th['a0'].copy()
    dz = D.d[:, None]

    def negf(g, a0):
        eta = (S @ g)[:, None] + a0[None, :]
        return -(W * np.where(dz, log_expit(eta), log_expit(-eta))).sum()

    f0 = negf(g, a0)
    for _ in range(n_newton):
        eta = (S @ g)[:, None] + a0[None, :]
        pr = expit(eta)
        R, V = W * (dz - pr), W * pr * (1 - pr)
        grad = np.concatenate([S.T @ R.sum(1), R.sum(0)])
        H = np.block([[(S * V.sum(1)[:, None]).T @ S, S.T @ V],
                      [(S.T @ V).T, np.diag(V.sum(0))]]) + 1e-8 * np.eye(p + G)
        step = np.linalg.solve(H, grad)
        t, ok = 1.0, False
        while t > 1e-4:
            gn = g + t * step[:p]
            an = np.clip(a0 + t * step[p:], -25, 25)
            f1 = negf(gn, an)
            if f1 <= f0 + 1e-10:
                ok = True
                break
            t /= 2
        if not ok:
            break
        g, a0, f0 = gn, an, f1
    return g, a0

def _posQ(D, W, tau, fam):
    """Objective (12) of the positive part and its gradient in (beta, alpha_1k), given sigma."""
    pos = ~D.d
    Xp, Wp = D.X[pos], W[pos]
    unc = D.delta[pos]
    cen = ~unc
    ly, lc = D.ly[pos], D.lc[pos]
    p = Xp.shape[1]

    def mu_of(theta):
        return (Xp @ theta[:p])[:, None] + theta[p:][None, :]

    def negQ(theta, s):
        mu = mu_of(theta)
        v = np.empty_like(mu)
        if unc.any():
            v[unc] = pos_logdens(fam, tau, ly[unc, None] - mu[unc], s)
        if cen.any():
            v[cen] = pos_logsurv(fam, tau, lc[cen, None] - mu[cen], s)
        return -(Wp * v).sum()

    def grad(theta, s):
        mu = mu_of(theta)
        gm = np.empty_like(mu)
        if unc.any():
            gm[unc] = pos_dmu_dens(fam, tau, ly[unc, None] - mu[unc], s)
        if cen.any():
            gm[cen] = pos_dmu_surv(fam, tau, lc[cen, None] - mu[cen], s)
        R = Wp * gm
        return -np.concatenate([Xp.T @ R.sum(1), R.sum(0)])

    return negQ, grad, p

def mstep_pos(D, th, W, tau, fam, maxiter=30):
    negQ, grad, p = _posQ(D, W, tau, fam)
    s0 = float(th['sigma'])
    th0 = np.concatenate([th['beta'], th['a1']])
    f0 = negQ(th0, s0)
    r = optimize.minimize(negQ, th0, args=(s0,), jac=grad, method='L-BFGS-B',
                          options={'maxiter': maxiter})
    theta = r.x if (np.isfinite(r.fun) and r.fun <= f0) else th0
    fs = lambda ls: negQ(theta, np.exp(ls))
    ls0 = np.log(s0)
    rs = optimize.minimize_scalar(fs, bounds=(ls0 - 3, ls0 + 3), method='bounded',
                                  options={'xatol': 1e-4})
    s = float(np.exp(rs.x)) if rs.fun <= fs(ls0) else s0
    return theta[:p], theta[p:], max(s, 1e-6)

def _mstep(D, th, w, tau, fam):
    th = dict(th)
    pi = np.clip(w.mean(0), 1e-10, None)
    th['pi'] = pi / pi.sum()
    W = w[D.uid]
    th['gamma'], th['a0'] = mstep_binary(D, th, W)
    th['beta'], th['a1'], th['sigma'] = mstep_pos(D, th, W, tau, fam)
    return th

def _em(D, th, tau, fam, maxit, tol):
    ll_old, it = -np.inf, 0
    for it in range(1, maxit + 1):
        w, ll = estep(D, th, tau, fam)
        if np.isfinite(ll_old) and abs(ll - ll_old) <= tol * abs(ll_old):
            return th, w, ll, it
        ll_old = ll
        th = _mstep(D, th, w, tau, fam)
    w, ll = estep(D, th, tau, fam)
    return th, w, ll, it

def _pooled_init(D, tau, fam):
    th = {'gamma': np.zeros(D.S.shape[1]), 'a0': np.zeros(1)}
    g, a0 = mstep_binary(D, th, np.ones((D.n, 1)), n_newton=25)
    unc = (~D.d) & D.delta
    Z = np.column_stack([np.ones(unc.sum()), D.X[unc]])
    coef = np.linalg.lstsq(Z, D.ly[unc], rcond=None)[0]
    r = D.ly[unc] - Z @ coef
    if fam == 'al':
        q = np.quantile(r, tau)
        s = max(np.mean(_rho(r - q, tau)), 1e-3)
    else:
        q, s = 0.0, max(r.std(), 1e-3)
    return g, a0[0], coef[1:], coef[0] + q, s

def _init_weights(D, G, rng, kind, seed):
    if G == 1:
        return np.ones((D.N, 1))
    if kind == 'kmeans':
        cnt = np.bincount(D.uid, minlength=D.N)
        zs = np.bincount(D.uid, weights=D.d.astype(float), minlength=D.N) / cnt
        pos = ~D.d
        npos = np.bincount(D.uid, weights=pos.astype(float), minlength=D.N)
        sly = np.bincount(D.uid, weights=np.where(pos, D.ly, 0.), minlength=D.N)
        mly = np.where(npos > 0, sly / np.maximum(npos, 1), D.ly[pos].mean())
        F = np.column_stack([zs, mly])
        F = (F - F.mean(0)) / (F.std(0) + 1e-12)
        try:
            lab = KMeans(G, n_init=5, random_state=seed).fit_predict(F)
            W = np.full((D.N, G), 0.1 / (G - 1))
            W[np.arange(D.N), lab] = 0.9
            return W
        except Exception:
            pass
    return rng.dirichlet(np.ones(G), size=D.N)

def _finalize(D, th, w, ll, G):
    o = np.argsort(th['a1'])
    th = dict(th)
    for k in ('a0', 'a1', 'pi'):
        th[k] = th[k][o]
    w = w[:, o]
    pi = th['pi']
    g0, b0 = float(pi @ th['a0']), float(pi @ th['a1'])          # centring, eq. (9)
    B = np.column_stack([th['a0'] - g0, th['a1'] - b0])
    Sb = (B * pi[:, None]).T @ B
    rho = (Sb[0, 1] / np.sqrt(Sb[0, 0] * Sb[1, 1]) if min(Sb[0, 0], Sb[1, 1]) > 1e-12 else np.nan)
    nu = D.S.shape[1] + D.X.shape[1] + 3 * G                     # = m + q + 3G - 2, eq. (13)
    bic = -2 * ll + nu * np.log(D.N)
    ent = -np.sum(w * np.log(np.clip(w, 1e-300, None)))
    return dict(gamma0=g0, gamma=th['gamma'].copy(), beta0=b0, beta=th['beta'].copy(),
                sigma=float(th['sigma']), pi=pi.copy(), b=B, Sigma_b=Sb, rho=rho,
                ll=ll, bic=bic, icl=bic + 2 * ent, G=G, nu=nu, theta=th, w=w, N=D.N, n=D.n,
                degenerate=bool(G > 1 and pi.min() < 0.01))

def _as_theta(v):
    return {k: (float(x) if k == 'sigma' else np.array(x, float)) for k, x in v.items()}

def fit(D, tau, G, fam='al', n_starts=3, seed=0, init=None, extra_starts=None,
        maxit=500, tol=1e-6, short_iter=20):
    """ECM fit at fixed tau and G. Multi-start: k-means, random, and optional
    user-supplied starts (e.g. the fit that ignores censoring)."""
    rng = np.random.default_rng(seed)
    t0 = time.time()
    if init is not None:
        th, w, ll, it = _em(D, _as_theta(init), tau, fam, maxit, tol)
    else:
        g, a0, b, a1, s = _pooled_init(D, tau, fam)
        n_short = short_iter if G > 1 else maxit
        cands = []
        for st in range(max(1, n_starts) if G > 1 else 1):
            W0 = _init_weights(D, G, rng, 'kmeans' if st == 0 else 'random', seed + st)
            th = {'gamma': g.copy(), 'a0': np.full(G, a0), 'beta': b.copy(),
                  'a1': np.full(G, a1), 'sigma': s, 'pi': W0.mean(0)}
            th = _mstep(D, th, W0, tau, fam)
            cands.append(_em(D, th, tau, fam, n_short, tol))
        for ex in (extra_starts or []):
            if ex is None or len(ex['a1']) != G:
                continue
            try:
                cands.append(_em(D, _as_theta(ex), tau, fam, n_short, tol))
            except Exception:
                pass
        th, w, ll, it = max(cands, key=lambda z: z[2] if np.isfinite(z[2]) else -np.inf)
        if G > 1:
            th, w, ll, it = _em(D, th, tau, fam, maxit, tol)
    res = _finalize(D, th, w, ll, G)
    res.update(tau=tau, fam=fam, iter=it, time=time.time() - t0)
    return res

def _safe_fit(*a, **k):
    try:
        return fit(*a, **k)
    except Exception:
        return None

def choose_G(fits, crit='bic'):
    """G minimising the criterion among non-degenerate fits (all fits if none)."""
    nd = {G: f for G, f in fits.items() if not f.get('degenerate', False)}
    pool = nd or fits
    return min(pool, key=lambda G: pool[G][crit])

def param_vector(r):
    return np.concatenate([[r['gamma0']], r['gamma'], [r['beta0']], r['beta'], [r['sigma']]])

def simulate_from(D, r, rng):
    th, G, tau, fam = r['theta'], r['G'], r['tau'], r['fam']
    k = rng.choice(G, size=D.N, p=th['pi'])[D.uid]
    zero = rng.random(D.n) < expit(D.S @ th['gamma'] + th['a0'][k])
    mu = D.X @ th['beta'] + th['a1'][k]
    e = (rng.exponential(size=D.n) / tau - rng.exponential(size=D.n) / (1 - tau)
         if fam == 'al' else rng.standard_normal(D.n))
    y = np.where(zero, 0.0, np.exp(np.clip(mu + th['sigma'] * e, -700, 700)))
    return PanelData(D.uid_raw, y, D.S, D.X, D.c)

def bootstrap(D, r, B=100, seed=1, method=BOOT_METHOD, n_jobs=N_JOBS):
    """Unit-level nonparametric (cluster) bootstrap, or parametric bootstrap.
    G is fixed at its selected value; refits start from the estimate."""
    k = param_vector(r).size
    rows_of = [np.flatnonzero(D.uid == u) for u in range(D.N)] if method == 'cluster' else None

    def one(b):
        rng = np.random.default_rng([int(seed), int(b)])
        try:
            if method == 'cluster':
                pick = rng.integers(0, D.N, D.N)
                idx = np.concatenate([rows_of[u] for u in pick])
                new_id = np.repeat(np.arange(D.N), [rows_of[u].size for u in pick])
                Db = PanelData(new_id, D.y[idx], D.S[idx], D.X[idx], D.c[idx])
            else:
                Db = simulate_from(D, r, rng)
            rb = fit(Db, r['tau'], r['G'], fam=r['fam'], init=r['theta'])
            return param_vector(rb)
        except Exception:
            return np.full(k, np.nan)

    P = np.array(Parallel(n_jobs=n_jobs)(delayed(one)(b) for b in range(B)))
    return dict(draws=P, se=np.nanstd(P, 0, ddof=1),
                lo=np.nanpercentile(P, 2.5, 0), hi=np.nanpercentile(P, 97.5, 0),
                B=B, method=method, n_failed=int(np.isnan(P[:, 0]).sum()))

# ----------------------------------------------------------------------
# 3. SIMULATION DESIGN
# ----------------------------------------------------------------------
GAM = np.array([-1.5, 1.0, -3.0])
BETA0 = 1.0
BETA = np.array([2.0, 0.5, -1.0])
SIG_B = np.array([[0.25, 0.0625], [0.0625, 0.25]])
TAUS_SIM = [0.1, 0.5, 0.9]
HET = 1.0                                  # E4: scale (1 + HET * x3)
ERR_LAB = {'E1': 'E1 (normal)', 'E2': r'E2 ($t_3$)', 'E3': r'E3 (skewed $\chi^2_3$)',
           'E4': 'E4 (heteroscedastic)'}
KAPPAS = [0.05, 0.20, 0.35]
G_SIM, NSTART_SIM = 3, 2
METHODS = ['M1', 'M2', 'M3', 'M4']

def err_draw(name, x3, rng):
    n = x3.size
    if name == 'E1':
        return 0.5 * rng.standard_normal(n)
    if name == 'E2':
        return 0.5 * rng.standard_t(3, n)
    if name == 'E3':
        return 0.5 * (rng.chisquare(3, n) - 3.0) / np.sqrt(6.0)
    if name == 'E4':
        return 0.5 * (1.0 + HET * x3) * rng.standard_normal(n)
    raise ValueError(name)

def err_q(name, tau):
    """tau-quantile of the error (for E4: at x3 = 0)."""
    if name in ('E1', 'E4'):
        return 0.5 * stats.norm.ppf(tau)
    if name == 'E2':
        return 0.5 * stats.t.ppf(tau, 3)
    if name == 'E3':
        return 0.5 * (stats.chi2.ppf(tau, 3) - 3.0) / np.sqrt(6.0)
    raise ValueError(name)

def truth(err, tau):
    """True tau-th conditional quantile coefficients of log y (given b, y > 0)."""
    b = BETA.copy()
    if err == 'E4':
        b[2] += HET * 0.5 * stats.norm.ppf(tau)
    return BETA0 + err_q(err, tau), b

_L = np.linalg.cholesky(SIG_B)
_ang = np.deg2rad([90, 210, 330])
R2_PTS = (np.sqrt(2) * np.column_stack([np.cos(_ang), np.sin(_ang)])) @ _L.T

def re_draw(kind, N, rng):
    if kind == 'R1':
        return rng.multivariate_normal(np.zeros(2), SIG_B, N)
    return R2_PTS[rng.integers(0, 3, N)]

def gen_raw(err, re_kind, N, T, gamma0, rng):
    n = N * T
    uid = np.repeat(np.arange(N), T)
    X = np.column_stack([rng.standard_normal(n), rng.standard_normal(n),
                         (rng.random(n) < 0.5).astype(float)])
    b = re_draw(re_kind, N, rng)[uid]
    zero = rng.random(n) < expit(gamma0 + X @ GAM + b[:, 0])
    ly = BETA0 + X @ BETA + b[:, 1] + err_draw(err, X[:, 2], rng)
    return uid, np.where(zero, 0.0, np.exp(ly)), X

def calibrate_gamma0(target=0.30, re_kind='R1', n=400_000, seed=7):
    rng = np.random.default_rng(seed)
    X = np.column_stack([rng.standard_normal(n), rng.standard_normal(n), rng.random(n) < .5])
    b0 = re_draw(re_kind, n, rng)[:, 0]
    return optimize.brentq(lambda g: expit(g + X @ GAM + b0).mean() - target, -10, 10)

def thresholds(err, re_kind, gamma0, kappas, n=1_000_000, seed=11):
    _, y, _ = gen_raw(err, re_kind, n, 1, gamma0, np.random.default_rng(seed))
    pos = y[y > 0]
    return {k: float(np.quantile(pos, 1 - k)) for k in kappas}

def sim_meta():
    fn = os.path.join(SIM_DIR, 'meta.json')
    meta = json.load(open(fn)) if os.path.exists(fn) else {}
    changed = False
    if 'gamma0' not in meta:
        meta['gamma0'] = calibrate_gamma0()
        changed = True
    thr = {e: {float(k): v for k, v in d.items()} for e, d in meta.get('thr', {}).items()}
    for e in ERRS_ALL:
        if e not in thr or any(k not in thr[e] for k in KAPPAS):
            thr[e] = thresholds(e, 'R1', meta['gamma0'], KAPPAS)
            changed = True
    meta['thr'] = thr
    if changed:
        json.dump({'gamma0': meta['gamma0'],
                   'thr': {e: {str(k): v for k, v in d.items()} for e, d in thr.items()}},
                  open(fn, 'w'), indent=2)
    return meta

def _row(m, tau, r, implied=False):
    out = dict(method=m, tau=tau)
    if r is None:
        return out
    b0 = r['beta0'] + (r['sigma'] * stats.norm.ppf(tau) if implied else 0.0)
    out.update(gamma0=r['gamma0'], beta0=b0, sigma=r['sigma'], ll=r['ll'], iter=r['iter'],
               sec=r['time'], rho=r['rho'], G=r['G'])
    out.update({f'gamma{j+1}': v for j, v in enumerate(r['gamma'])})
    out.update({f'beta{j+1}': v for j, v in enumerate(r['beta'])})
    return out

def sim_rep(err, N, T, kappa, rep, gamma0, cthr):
    seed = zlib.crc32(f'{err}_R1_{N}_{T}_{kappa:.2f}_{rep}'.encode())
    uid, y, X = gen_raw(err, 'R1', N, T, gamma0, np.random.default_rng(seed))
    D = PanelData(uid, y, X, X, cthr)
    D2, D3 = D.naive(), D.subset(D.delta)
    base = dict(err=err, re='R1', N=N, T=T, kappa=kappa, rep=rep, zero_share=D.d.mean(),
                cens_share=(~D.delta).sum() / max((~D.d).sum(), 1))
    s = seed % 10**6
    rows = []
    for tau in TAUS_SIM:
        r2 = _safe_fit(D2, tau, G_SIM, 'al', NSTART_SIM, s)
        r1 = _safe_fit(D, tau, G_SIM, 'al', NSTART_SIM, s, None, [r2['theta']] if r2 else None)
        r3 = _safe_fit(D3, tau, G_SIM, 'al', NSTART_SIM, s)
        for m, r in (('M1', r1), ('M2', r2), ('M3', r3)):
            rows.append({**base, **_row(m, tau, r)})
    r4 = _safe_fit(D, 0.5, G_SIM, 'gauss', NSTART_SIM, s)
    for tau in TAUS_SIM:                    # implied conditional quantiles of the Tobit model
        rows.append({**base, **_row('M4', tau, r4, implied=True)})
    return rows

def sim_file(err, N, T, k):
    return os.path.join(SIM_DIR, f'{err}_R1_N{N}_T{T}_k{k:.2f}.csv')

def run_sim_cell(err, N, T, kappa, gamma0, cthr, B):
    fn = sim_file(err, N, T, kappa)
    done = set(pd.read_csv(fn)['rep']) if os.path.exists(fn) else set()
    todo = [r for r in range(B) if r not in done]
    if not todo:
        return
    batch = max(4, 4 * NCPU)
    for i in range(0, len(todo), batch):
        chunk = todo[i:i + batch]
        t0 = time.time()
        res = Parallel(n_jobs=N_JOBS)(delayed(sim_rep)(err, N, T, kappa, r, gamma0, cthr) for r in chunk)
        pd.DataFrame([x for rr in res for x in rr]).to_csv(fn, mode='a', header=not os.path.exists(fn),
                                                           index=False)
        n_done = len([r for r in range(B) if r in done]) + min(i + batch, len(todo))
        print(f'  {os.path.basename(fn)}: {n_done}/{B} ({time.time() - t0:.0f} s)', flush=True)

def sim_progress(errs, B):
    tot = 0
    for e in errs:
        for (N, T) in M['sim_nt']:
            for k in KAPPAS:
                fn = sim_file(e, N, T, k)
                if os.path.exists(fn):
                    r = pd.read_csv(fn, usecols=['rep'])['rep']
                    tot += r[r < B].nunique()
    full = len(errs) * len(M['sim_nt']) * len(KAPPAS) * B